# NeRF: volume rendering a neural radiance field

*graphics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/nerf_volume_rendering)

A small MLP $F_\theta(\mathbf x) = (\sigma, \mathbf c)$ gives every point in space a density and a colour. A pixel is the light collected along its ray: stratified samples $t_s$, then the quadrature $C = \sum_s T_s\,(1 - e^{-\sigma_s\delta_s})\,\mathbf c_s$ with transmittance $T_s = e^{-\sum_{s'<s}\sigma_{s'}\delta_{s'}}$, batched over rays × samples. Trained only on photos of a synthetic scene, it renders a held-out view and a depth map, and a second network without positional encoding shows the spectral bias of plain MLPs.

**Vector form**

$$C(\mathbf r) = \int_{t_n}^{t_f} T(t)\,\sigma(\mathbf r(t))\,\mathbf c(\mathbf r(t))\,dt,\quad T(t) = e^{-\int_{t_n}^{t}\sigma(\mathbf r(u))\,du},\quad \mathbf r(t) = \mathbf o + t\,\mathbf d,\qquad \min_\theta \sum_{\mathbf r}\lVert C_\theta(\mathbf r) - C^\ast(\mathbf r)\rVert^2$$

**Index (tensor) form**

$$x_{rsi} = o_{ri} + t_{rs}\,d_{ri},\qquad (\sigma_{rs}, c_{rsc}) = F_\theta\big(\gamma(x_{rs\cdot})\big),\qquad w_{rs} = e^{-\sum_{s'<s}\sigma_{rs'}\delta_{rs'}}\big(1 - e^{-\sigma_{rs}\delta_{rs}}\big),\qquad C_{rc} = w_{rs}\,c_{rsc} + \big(1 - \textstyle\sum_s w_{rs}\big)\,b_c$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $o_{ri},\ d_{ri}$ | `cameras(azimuths, elevations, res)` | (V, P, 3) | origin and unit direction of the ray through each of the $P$ pixels of each camera |
| $t_{rs}$ | `stratified_samples(R, S, near, far, gen)` | (R, S) | one random depth in each of $S$ equal bins of $[t_n, t_f]$: a new set every step, so the network sees the whole ray |
| $x_{rsi}$ | `o[:, None] + t[..., None] * d[:, None]` | (R, S, 3) | sample points: broadcasting over rays $r$ and samples $s$ |
| $\gamma(x)_{rsik}$ | `einsum('...i,k->...ik', x, freq)` | (R, S, 3+6L) | $\sin$ and $\cos$ of $2^k\pi x_i$, $k = 0 \dots L-1$; nothing summed, an outer product |
| $A_{og}\,h_{rsg} + b_o$ | `einsum('og,...g->...o', A, h) + b` | (R, S, H) | one MLP layer applied to every sample at once, $g$ summed; ReLU between layers |
| $\sigma_{rs},\ c_{rsc}$ | `softplus(out[..., 0] - 1), sigmoid(out[..., 1:])` | (R, S), (R, S, 3) | density $\ge 0$ and colour in $(0, 1)$ |
| $\delta_{rs}$ | `cat([t[:, 1:] - t[:, :-1], far - t[:, -1:]], 1)` | (R, S) | length of each interval along the ray |
| $T_{rs}$ | `exp(-cumsum(tau, 1) + tau)` | (R, S) | transmittance: an exclusive cumulative sum of the optical thickness $\tau = \sigma\delta$ over $s' < s$ |
| $w_{rs}$ | `T * (1 - exp(-tau))` | (R, S) | probability that the light reaching the pixel was emitted in interval $s$; $\sum_s w_{rs} \le 1$ |
| $C_{rc} = w_{rs}c_{rsc} + \dots$ | `einsum('rs,rsc->rc', w, colour) + (1 - acc) * bg` | (R, 3) | pixel colour, $s$ summed; the left-over transmittance shows the white background |
| $D_r = w_{rs}\,t_{rs}$ | `einsum('rs,rs->r', w, t)` | (R,) | expected depth: the same weights give a depth map for free |

The training data are $V$ photos of a red sphere, a checkered cube and an orange ball, made by an analytic ray caster with Lambertian shading; the network never sees the geometry, only pixel colours and the ray of each pixel. Each Adam step draws a random batch of rays from all training pixels, renders them with the quadrature and compares with the true colours. Because $w_{rs}$, $T_{rs}$ and the colours are smooth functions of $\sigma$ and $\mathbf c$, autograd carries the photo error back into the network: densities grow where the rays of many views agree on a surface. The depth map $\sum_s w_s t_s$ and the weight profile along the marked ray show it: the weights collapse to a peak at the true surface. Positional encoding is the key trick. A plain MLP on $(x, y, z)$ learns low frequencies first and fits sharp edges and fine texture very slowly (spectral bias); feeding it $\sin(2^k\pi x), \cos(2^k\pi x)$ instead gives it high-frequency building blocks. In the same number of steps the encoded network (blue) has crisper silhouettes, about half the depth error and a higher PSNR on the training views, while the plain one (red) stays blurred. On the held-out view the gain is smaller: with few, small views the fine detail is only weakly constrained, and both networks are still far from converged (real NeRFs train for hours). Simplifications versus Mildenhall et al. (ECCV 2020): no view direction input (the scene is Lambertian, so its colours do not depend on the viewing angle), one network instead of the coarse + fine pair with hierarchical sampling, a tiny MLP (3 × 64 instead of 8 × 256), $40\times 40$ images, and a few hundred steps instead of hundreds of thousands. Try $L = 2$ or $L = 10$, fewer samples $S$ (banding from the coarse quadrature), more steps (the checkerboard appears with encoding only), or fewer views (the geometry becomes ambiguous and the held-out view degrades while the training views still fit).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/nerf_volume_rendering/main.py
"""Main loop: analytic scene → training photos → train two radiance fields (with / without encoding) → novel view."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors

SEED = 0
DEPTH = 3  # hidden layers
N_SNAP = 25


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    res, n_views, n_freq = params["resolution"], params["n_views"], params["n_freq"]
    n_steps = 30 if fast else params["steps"]
    n_samples, batch = params["n_samples"], params["batch"]
    torch.manual_seed(SEED)

    sc = tensors.scene(dev)
    cams = tensors.training_cameras(n_views, res, dev)
    o, d = cams["origins"].reshape(-1, 3), cams["dirs"].reshape(-1, 3)  # (V·P, 3) every training ray
    target, _ = tensors.render_scene(sc, o, d)  # (V·P, 3) the training photos, pixel by pixel
    nov = tensors.novel_camera(n_views, res, dev)
    o_n, d_n = nov["origins"][0], nov["dirs"][0]  # (P, 3)
    rgb_n, t_true = tensors.render_scene(sc, o_n, d_n)  # held-out photo and true depth
    print(f"scene: sphere + checkered cube + ball, {n_views} training views of {res}×{res} px = {o.shape[0]:,} "
          f"rays; sampled on t ∈ [{tensors.NEAR:.1f}, {tensors.FAR:.1f}] with {n_samples} samples each")  # fmt: skip

    runs = {}
    for name, L in [("with encoding", n_freq), ("without encoding", 0)]:
        in_dim = 3 + 6 * L
        layers = tensors.init_mlp(in_dim, params["width"], DEPTH, SEED, dev)
        n_par = sum(p.numel() for p in tensors.parameters(layers))
        t0 = time.perf_counter()
        hist = solve.train(layers, L, o, d, target, (o_n, d_n, rgb_n), n_steps, params["lr"], batch, n_samples,
                           5 if fast else N_SNAP, SEED, label=f"steps ({name})")  # fmt: skip
        dt = time.perf_counter() - t0
        last = hist["snaps"][-1]
        hit = torch.isfinite(t_true).cpu()
        depth_err = (last["depth"][hit] - t_true.cpu()[hit]).abs().median().item()
        print(f"{name:>17} (L = {L}, input {in_dim:2d} → {DEPTH}×{params['width']} → 4, {n_par:,} weights): "
              f"{n_steps} steps × {batch} rays in {dt:.1f} s")  # fmt: skip
        print(f"{'':>19}PSNR training view {last['psnr_train']:.1f} dB, held-out {hist['snaps'][0]['psnr']:.1f} → "
              f"{last['psnr']:.1f} dB, median depth error {depth_err:.3f}")  # fmt: skip
        runs[name] = hist

    t_far = torch.where(torch.isfinite(t_true), t_true, torch.full_like(t_true, tensors.FAR))
    return plot.animate(rgb_n.cpu(), t_far.cpu(), runs, res, n_freq, out_dir / "nerf_volume_rendering.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/nerf_volume_rendering/equation.py
r"""Physics → tensor translation.

A neural radiance field (NeRF) is a function F_θ: x ↦ (σ, c) that gives every point x in space a density σ ≥ 0
(how much light it blocks per unit length) and a colour c. A camera pixel sees the light collected along its ray
x(t) = o + t d. Indices: r = ray, s = sample along the ray, i = space axis, k = frequency, h = hidden unit.

    encoding    γ(x)_{rsik} = (sin(2^k π x_{rsi}), cos(2^k π x_{rsi})),   k = 0 … L-1
    network     (σ_{rs}, c_{rs·}) = MLP(γ(x_{rs}))      layers h_{n+1} = ReLU(A_{hg} h_g + b_h), g summed
    opacity     α_{rs} = 1 - exp(-σ_{rs} δ_{rs}),   δ_{rs} = t_{r,s+1} - t_{rs}
    transmit.   T_{rs} = exp(-Σ_{s' < s} σ_{rs'} δ_{rs'})                 light that survives up to sample s
    colour      C_r = Σ_s T_{rs} α_{rs} c_{rs} + (1 - Σ_s T_{rs} α_{rs}) c_bg      s summed: w_{rs} = T_{rs} α_{rs}

The last line is the quadrature of the volume rendering integral C = ∫ T(t) σ(t) c(t) dt, exact when σ and c are
constant on each interval. Everything is differentiable, so the network is trained from photos alone.
"""

import math

import torch
import torch.nn.functional as F


def positional_encoding(x: torch.Tensor, n_freq: int) -> torch.Tensor:
    """γ(x) = [x, sin(2^k π x), cos(2^k π x)]_{k < L}: (..., 3) → (..., 3 + 6L). With L = 0 it returns x itself.

    The outer product x_{…i} 2^k π is an einsum with no summed index ('…i,k->…ik').
    """
    if n_freq == 0:
        return x
    freq = math.pi * 2.0 ** torch.arange(n_freq, device=x.device, dtype=x.dtype)  # (L,)
    xk = torch.einsum("...i,k->...ik", x, freq).flatten(-2)  # (..., 3L)
    return torch.cat([x, torch.sin(xk), torch.cos(xk)], dim=-1)


def mlp(h: torch.Tensor, layers: list[dict]) -> torch.Tensor:
    """h_{n+1} = ReLU(A_{og} h_{…g} + b_o)   (g summed), no ReLU on the output.   (..., D) → (..., 4)."""
    for n, layer in enumerate(layers):
        h = torch.einsum("og,...g->...o", layer["A"], h) + layer["b"]
        if n < len(layers) - 1:
            h = F.relu(h)
    return h


def radiance_field(x: torch.Tensor, layers: list[dict], n_freq: int, scale: float):
    """F_θ(x) → density σ (...,) ≥ 0 via softplus, colour c (..., 3) ∈ (0, 1) via sigmoid."""
    out = mlp(positional_encoding(x / scale, n_freq), layers)
    return F.softplus(out[..., 0] - 1.0), torch.sigmoid(out[..., 1:])


def stratified_samples(n_rays: int, n_samples: int, near: float, far: float, gen, device) -> torch.Tensor:
    """t_{rs}: one uniform random point in each of S equal bins of [near, far], per ray.   → (R, S).

    gen = None gives the bin midpoints (deterministic, used for evaluation).
    """
    edges = torch.linspace(near, far, n_samples + 1, device=device)
    width = edges[1] - edges[0]
    u = 0.5 * torch.ones(n_rays, n_samples) if gen is None else torch.rand(n_rays, n_samples, generator=gen)
    return edges[:-1] + width * u.to(device)


def render_weights(sigma: torch.Tensor, t: torch.Tensor, far: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Quadrature weights w_{rs} = T_{rs} (1 - e^{-σ_{rs} δ_{rs}}) and transmittance T_{rs}.   (R, S), (R, S).

    δ_{rs} = t_{r,s+1} - t_{rs}; the last interval runs to `far`.
    """
    delta = torch.cat([t[:, 1:] - t[:, :-1], far - t[:, -1:]], dim=1)  # (R, S)
    tau = sigma * delta  # optical thickness of each interval
    T = torch.exp(-torch.cumsum(tau, dim=1) + tau)  # exclusive sum: Σ_{s' < s}
    return T * (1 - torch.exp(-tau)), T


def volume_render(sigma, colour, t, far, background):
    """C_r = Σ_s w_{rs} c_{rsc} + (1 - Σ_s w_{rs}) b_c   (s summed), and the expected depth D_r = Σ_s w_{rs} t_{rs}.

    sigma (R, S), colour (R, S, 3), t (R, S) → colour (R, 3), depth (R,), weights (R, S).
    """
    w, _ = render_weights(sigma, t, far)
    acc = w.sum(dim=1)  # (R,) total opacity of the ray
    rgb = torch.einsum("rs,rsc->rc", w, colour) + (1 - acc)[:, None] * background
    depth = torch.einsum("rs,rs->r", w, t) + (1 - acc) * far  # an empty ray counts as 'at the far plane'
    return rgb, depth, w


def render_rays(o, d, layers, n_freq, scale, n_samples, near, far, background, gen=None):
    """Whole pipeline for a batch of rays o, d (R, 3): samples → field → quadrature → colour (R, 3), depth (R,)."""
    t = stratified_samples(o.shape[0], n_samples, near, far, gen, o.device)  # (R, S)
    x = o[:, None, :] + t[..., None] * d[:, None, :]  # (R, S, 3) sample points x_{rsi}
    sigma, colour = radiance_field(x, layers, n_freq, scale)
    rgb, depth, w = volume_render(sigma, colour, t, far, background)
    return rgb, depth, w, t


def psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Peak signal-to-noise ratio in dB for images in [0, 1]: -10 log₁₀ MSE."""
    return -10 * torch.log10(((pred - target) ** 2).mean())


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/nerf_volume_rendering/tensors.py
"""Build the concrete tensors: the analytic scene, the cameras and their rays, the true images and the MLP weights."""

import math

import torch

BACKGROUND = (1.0, 1.0, 1.0)  # white
LIGHT = (0.45, -0.35, 0.82)  # direction towards the light (world frame, z up)
AMBIENT = 0.35
RADIUS = 3.6  # camera distance from the origin
FOV = 30.0  # full field of view in degrees
NEAR, FAR = RADIUS - 1.6, RADIUS + 1.6  # every ray is sampled on t ∈ [NEAR, FAR]
SCALE = 1.5  # positions are divided by this before encoding, so the scene fits in [-1, 1]³


def scene(device: torch.device) -> dict:
    """A red sphere next to a checkered cube (rotated about z) and a small orange ball on top of the cube.

    The checkerboard is the high-frequency detail that a plain MLP struggles to learn.
    """
    return {
        "centres": torch.tensor([[-0.45, -0.25, -0.1], [0.4, 0.3, 0.42]], device=device),
        "radii": torch.tensor([0.5, 0.2], device=device),
        "sphere_rgb": torch.tensor([[0.86, 0.15, 0.15], [0.96, 0.55, 0.04]], device=device),
        "box_centre": torch.tensor([0.4, 0.3, -0.15], device=device),
        "box_half": torch.tensor([0.37, 0.37, 0.37], device=device),
        "box_yaw": 0.6,  # rad
        "box_rgb": torch.tensor([0.09, 0.64, 0.29], device=device),  # checkerboard colours
        "box_rgb2": torch.tensor([0.93, 0.93, 0.85], device=device),
    }


def cameras(azimuths: list[float], elevations: list[float], res: int, device: torch.device) -> dict:
    """V pinhole cameras on a sphere of radius RADIUS looking at the origin; returns the ray of every pixel:

    origins (V, P, 3), unit directions (V, P, 3), with P = res² pixels in row-major order.
    """
    az = torch.tensor(azimuths, device=device)
    el = torch.tensor(elevations, device=device)
    centre = RADIUS * torch.stack([el.cos() * az.cos(), el.cos() * az.sin(), el.sin()], dim=-1)  # (V, 3)
    forward = -centre / centre.norm(dim=-1, keepdim=True)
    up = torch.tensor([0.0, 0.0, 1.0], device=device).expand_as(forward)
    right = torch.linalg.cross(forward, up)
    right = right / right.norm(dim=-1, keepdim=True)
    down = torch.linalg.cross(forward, right)
    f = 0.5 * res / math.tan(math.radians(FOV / 2))
    ij = (torch.arange(res, device=device, dtype=torch.float32) + 0.5 - res / 2) / f
    v, u = torch.meshgrid(ij, ij, indexing="ij")
    u, v = u.reshape(-1), v.reshape(-1)  # (P,)
    d = right[:, None] * u[None, :, None] + down[:, None] * v[None, :, None] + forward[:, None]  # (V, P, 3)
    d = d / d.norm(dim=-1, keepdim=True)
    return {"origins": centre[:, None, :].expand_as(d).contiguous(), "dirs": d, "res": res}


def training_cameras(n_views: int, res: int, device: torch.device) -> dict:
    """n_views cameras evenly spaced in azimuth, alternating between 5° and 45° elevation."""
    az = [2 * math.pi * k / n_views for k in range(n_views)]
    el = [math.radians(5.0 if k % 2 == 0 else 45.0) for k in range(n_views)]
    return cameras(az, el, res, device)


def novel_camera(n_views: int, res: int, device: torch.device) -> dict:
    """A held-out view between two training cameras, at 25° elevation."""
    return cameras([math.pi / n_views + 0.9], [math.radians(25.0)], res, device)


def render_scene(sc: dict, o: torch.Tensor, d: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Ground truth by analytic ray casting (spheres + cube, Lambertian shading, white background).

    o, d: (..., 3) → colour (..., 3) and hit distance t (..., ), inf where the ray misses.
    """
    t_best = torch.full(o.shape[:-1], float("inf"), device=o.device)
    normal = torch.zeros_like(o)
    rgb = torch.zeros_like(o)
    for c, r, col in zip(sc["centres"], sc["radii"], sc["sphere_rgb"]):  # |o + t d - c|² = r²
        oc = o - c
        b = (oc * d).sum(-1)
        disc = b**2 - (oc * oc).sum(-1) + r**2
        t = -b - disc.clamp(min=0).sqrt()
        hit = (disc > 0) & (t > 0) & (t < t_best)
        t_best = torch.where(hit, t, t_best)
        normal = torch.where(hit[..., None], (o + t[..., None] * d - c) / r, normal)
        rgb = torch.where(hit[..., None], col, rgb)

    cy, sy = math.cos(sc["box_yaw"]), math.sin(sc["box_yaw"])
    Rb = torch.tensor([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]], device=o.device)  # box frame → world
    ob = (o - sc["box_centre"]) @ Rb  # Rᵀ(o - c), row vectors
    db = d @ Rb
    h = sc["box_half"]
    t1, t2 = (-h - ob) / db, (h - ob) / db  # slab test
    t_near = torch.minimum(t1, t2).amax(-1)
    t_far = torch.maximum(t1, t2).amin(-1)
    hit = (t_far > t_near) & (t_near > 0) & (t_near < t_best)
    p = (ob + t_near[..., None] * db) / h
    face = p.abs().argmax(-1, keepdim=True)
    nb = torch.zeros_like(p).scatter(-1, face, p.gather(-1, face).sign())
    t_best = torch.where(hit, t_near, t_best)
    normal = torch.where(hit[..., None], nb @ Rb.T, normal)
    checks = (torch.floor(p + 1).sum(-1) % 2)[..., None]  # 2 × 2 checkerboard on every face
    box_rgb = checks * sc["box_rgb"] + (1 - checks) * sc["box_rgb2"]
    rgb = torch.where(hit[..., None], box_rgb, rgb)

    light = torch.tensor(LIGHT, device=o.device)
    shade = AMBIENT + (1 - AMBIENT) * (normal @ (light / light.norm())).clamp(min=0)
    colour = torch.where(torch.isfinite(t_best)[..., None], rgb * shade[..., None], torch.tensor(BACKGROUND).to(o))
    return colour, t_best


def init_mlp(in_dim: int, width: int, depth: int, seed: int, device: torch.device) -> list[dict]:
    """Weights of an MLP in_dim → width → … → width → 4 (σ, r, g, b): `depth` hidden layers, He-uniform init."""
    gen = torch.Generator().manual_seed(seed)
    sizes = [in_dim] + [width] * depth + [4]
    layers = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        bound = math.sqrt(6.0 / n_in)
        A = (torch.rand(n_out, n_in, generator=gen) * 2 - 1) * bound
        layers.append({"A": A.to(device).requires_grad_(True), "b": torch.zeros(n_out, device=device).requires_grad_()})
    return layers


def parameters(layers: list[dict]) -> list[torch.Tensor]:
    return [t for layer in layers for t in layer.values()]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/nerf_volume_rendering/solve.py
"""Train a radiance field on random batches of rays (Adam on the pixel MSE) and render whole held-out images."""

import torch

from problems._common.progress import steps

from .equation import psnr, render_rays
from .tensors import BACKGROUND, FAR, NEAR, SCALE, parameters


def render_image(layers, n_freq, o, d, n_samples, chunk=2048):
    """Render every ray of one image (P rays) in chunks, with bin-centre samples: colour (P, 3), depth (P,)."""
    bg = torch.tensor(BACKGROUND, device=o.device)
    out_rgb, out_depth = [], []
    with torch.no_grad():
        for start in range(0, o.shape[0], chunk):
            rgb, depth, _, _ = render_rays(o[start : start + chunk], d[start : start + chunk], layers, n_freq, SCALE,
                                           n_samples, NEAR, FAR, bg)  # fmt: skip
            out_rgb.append(rgb)
            out_depth.append(depth)
    return torch.cat(out_rgb), torch.cat(out_depth)


def train(layers, n_freq, o, d, target, novel, n_steps, lr, batch, n_samples, n_snap, seed, label="Adam steps"):
    """Adam on the mean squared error of random ray batches drawn from all training pixels.

    o, d, target: (N_rays, 3) every training ray and its true colour; the first P of them are training view 0.
    novel = (o, d, rgb) of the held-out image, P rays. Returns the loss per step and, at n_snap checkpoints, the
    held-out render, its depth map and PSNR, the PSNR of training view 0, and the weights w_s along the held-out
    view's centre ray.
    """
    gen = torch.Generator().manual_seed(seed)
    bg = torch.tensor(BACKGROUND, device=o.device)
    opt = torch.optim.Adam(parameters(layers), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps, eta_min=lr / 10)
    snap_at = set(torch.linspace(0, n_steps, n_snap).round().long().tolist())
    o_n, d_n, rgb_n = novel
    n_pix = o_n.shape[0]
    centre = o_n.shape[0] // 2 + int(o_n.shape[0] ** 0.5) // 2  # the middle pixel of the image
    losses, snaps = [], []

    def snapshot(it: int):
        img, depth = render_image(layers, n_freq, o_n, d_n, n_samples)
        img_train, _ = render_image(layers, n_freq, o[:n_pix], d[:n_pix], n_samples)
        with torch.no_grad():
            _, _, w, t = render_rays(o_n[[centre]], d_n[[centre]], layers, n_freq, SCALE, n_samples, NEAR, FAR, bg)
        snaps.append({"it": it, "rgb": img.cpu(), "depth": depth.cpu(), "psnr": psnr(img, rgb_n).item(),
                      "psnr_train": psnr(img_train, target[:n_pix]).item(),
                      "w": w[0].cpu(), "t": t[0].cpu()})  # fmt: skip

    for it in steps(n_steps, label):
        if it in snap_at:
            snapshot(it)
        idx = torch.randint(0, o.shape[0], (batch,), generator=gen).to(o.device)  # random rays from all views
        rgb, _, _, _ = render_rays(o[idx], d[idx], layers, n_freq, SCALE, n_samples, NEAR, FAR, bg, gen)
        loss = ((rgb - target[idx]) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
    snapshot(n_steps)
    return {"loss": torch.tensor(losses), "snaps": snaps}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/nerf_volume_rendering/plot.py
"""Animate training: held-out view and depth map with and without positional encoding, PSNR, and ray weights."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import save_gif

from .tensors import FAR, NEAR

BLUE, RED, GREY, DARK = "#2563eb", "#dc2626", "#9ca3af", "#374151"


def _img(x: torch.Tensor, res: int) -> np.ndarray:
    """(P, 3) colours → (res, res, 3) image; (P,) depths → (res, res) map."""
    return x.reshape(res, res, 3).clamp(0, 1).numpy() if x.dim() == 2 else x.reshape(res, res).numpy()


def animate(rgb_true: torch.Tensor, depth_true: torch.Tensor, runs: dict, res: int, n_freq: int, path: Path) -> Path:
    pe, raw = runs["with encoding"]["snaps"], runs["without encoding"]["snaps"]
    its = np.array([s["it"] for s in pe])
    p_pe, p_raw = np.array([s["psnr"] for s in pe]), np.array([s["psnr"] for s in raw])
    q_pe, q_raw = np.array([s["psnr_train"] for s in pe]), np.array([s["psnr_train"] for s in raw])
    d_lo, d_hi = float(depth_true.min()) - 0.1, FAR
    centre = res * res // 2 + res // 2
    t_surface = float(depth_true[centre])

    fig, axes = plt.subplots(2, 4, figsize=(8.6, 4.8))
    titles = ["held-out view: truth", f"NeRF, encoding L = {n_freq}", "NeRF, no encoding",
              "true depth", "depth $\\sum_s w_s t_s$", "depth, no encoding"]  # fmt: skip
    for ax, title in zip([*axes[:, 0], *axes[:, 1], *axes[:, 2]], [titles[i] for i in (0, 3, 1, 4, 2, 5)]):
        ax.set_title(title, fontsize=8.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    axes[0, 0].imshow(_img(rgb_true, res))
    axes[1, 0].imshow(_img(depth_true, res), cmap="viridis_r", vmin=d_lo, vmax=d_hi)
    im_pe = axes[0, 1].imshow(_img(pe[0]["rgb"], res))
    im_raw = axes[0, 2].imshow(_img(raw[0]["rgb"], res))
    dm_pe = axes[1, 1].imshow(_img(pe[0]["depth"], res), cmap="viridis_r", vmin=d_lo, vmax=d_hi)
    dm_raw = axes[1, 2].imshow(_img(raw[0]["depth"], res), cmap="viridis_r", vmin=d_lo, vmax=d_hi)
    axes[1, 0].plot([res // 2], [res // 2], "+", color=RED, ms=7)  # the ray shown bottom right

    ax_p = axes[0, 3]
    (l_pe,) = ax_p.plot([], [], "o-", color=BLUE, ms=2.5, lw=1.2, label=f"encoding L = {n_freq}")
    (l_raw,) = ax_p.plot([], [], "s-", color=RED, ms=2.5, lw=1.2, label="no encoding")
    (k_pe,) = ax_p.plot([], [], "--", color=BLUE, lw=1, label="training view")
    (k_raw,) = ax_p.plot([], [], "--", color=RED, lw=1)
    ax_p.set_xlim(0, its[-1])
    ax_p.set_ylim(min(p_pe.min(), p_raw.min()) - 1, max(q_pe.max(), q_raw.max(), p_pe.max()) + 3)
    ax_p.set_title("PSNR (dB): held-out, - - training", fontsize=8.5)
    ax_p.set_xlabel("Adam step", fontsize=7.5)
    ax_p.tick_params(labelsize=7)
    ax_p.legend(fontsize=6, loc="lower right")

    ax_w = axes[1, 3]
    t = pe[0]["t"].numpy()
    w_max = max(max(s["w"].max().item() for s in pe), max(s["w"].max().item() for s in raw))
    ax_w.axvline(t_surface, color=GREY, ls="--", lw=1)
    ax_w.text(t_surface, w_max * 1.05, " surface", fontsize=6.5, color="#6b7280", va="top")
    (w_pe,) = ax_w.plot(t, pe[0]["w"].numpy(), color=BLUE, lw=1.3)
    (w_raw,) = ax_w.plot(t, raw[0]["w"].numpy(), color=RED, lw=1.1)
    ax_w.set_xlim(NEAR, FAR)
    ax_w.set_ylim(0, w_max * 1.1)
    ax_w.set_title("weights $w_s$ on the + ray", fontsize=8.5)
    ax_w.set_xlabel("distance $t$ along the ray", fontsize=7.5)
    ax_w.tick_params(labelsize=7)
    counter = fig.text(0.5, 0.01, "", ha="center", va="bottom", fontsize=9)

    def update(k: int):
        im_pe.set_data(_img(pe[k]["rgb"], res))
        im_raw.set_data(_img(raw[k]["rgb"], res))
        dm_pe.set_data(_img(pe[k]["depth"], res))
        dm_raw.set_data(_img(raw[k]["depth"], res))
        l_pe.set_data(its[: k + 1], p_pe[: k + 1])
        l_raw.set_data(its[: k + 1], p_raw[: k + 1])
        k_pe.set_data(its[: k + 1], q_pe[: k + 1])
        k_raw.set_data(its[: k + 1], q_raw[: k + 1])
        w_pe.set_ydata(pe[k]["w"].numpy())
        w_raw.set_ydata(raw[k]["w"].numpy())
        counter.set_text(
            f"step {its[k]}   ·   held-out PSNR: with encoding {p_pe[k]:.1f} dB, without {p_raw[k]:.1f} dB"
        )

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, list(range(len(pe))), path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_freq': 6,
    'width': 64,
    'n_samples': 32,
    'steps': 600,
    'batch': 512,
    'lr': 0.005,
    'n_views': 24,
    'resolution': 40
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.nerf_volume_rendering")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.nerf_volume_rendering.main")
problem_dir = Path("problems/graphics/nerf_volume_rendering")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
